# Importing Dependencies

In [1]:
import pandas as pd
import numpy as np

# Loading Dataset & Getting Basic Understanding

In [2]:
# Load dataset
df = pd.read_csv("../data/raw/netflix_titles.csv", encoding = 'utf-8', on_bad_lines = 'skip')

# Basic Inspection
print(df.shape)
df.head()

(8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


In [4]:
df.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

# Data Cleaning

In [5]:
# Handling Missing Values
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')

In [6]:
df.isnull().sum()

show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      10
release_year     0
rating           4
duration         3
listed_in        0
description      0
dtype: int64

In [7]:
# Converting date_added to datetime
df['date_added'] = pd.to_datetime(df['date_added'], errors = 'coerce')

In [8]:
df['date_added'].info()

<class 'pandas.Series'>
RangeIndex: 8807 entries, 0 to 8806
Series name: date_added
Non-Null Count  Dtype         
--------------  -----         
8709 non-null   datetime64[us]
dtypes: datetime64[us](1)
memory usage: 68.9 KB


# Feature Engineering

In [9]:
# Extract new columns
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

In [10]:
# Clean Duration - True = TV Show, False = Movie
df['duration_int'] = df['duration'].str.extract('(\d+)').astype(float)
df['duration_type'] = df['duration'].str.contains('Season', na = False)

In [11]:
# Split Genres
df['primary_genre'] = df['listed_in'].apply(lambda x: x.split(',')[0])

In [12]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,duration_int,duration_type,primary_genre
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021.0,9.0,90.0,False,Documentaries
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021.0,9.0,2.0,True,International TV Shows
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021.0,9.0,1.0,True,Crime TV Shows
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021.0,9.0,1.0,True,Docuseries
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021.0,9.0,2.0,True,International TV Shows


In [13]:
# Clean Release Year
df['content_age'] = 2026 - df['release_year']

In [14]:
# Cleaned Data
df.to_csv("../data/cleaned/netflix_cleaned.csv", index = False, encoding = 'utf-8', quoting = 1)

In [15]:
print('Type')
print(df['type'].value_counts())

print('Primary Genre')
print(df['primary_genre'].value_counts())

Type
type
Movie      6131
TV Show    2676
Name: count, dtype: int64
Primary Genre
primary_genre
Dramas                          1600
Comedies                        1210
Action & Adventure               859
Documentaries                    829
International TV Shows           774
Children & Family Movies         605
Crime TV Shows                   399
Kids' TV                         388
Stand-Up Comedy                  334
Horror Movies                    275
British TV Shows                 253
Docuseries                       221
Anime Series                     176
International Movies             128
TV Comedies                      120
Reality TV                       120
Classic Movies                    80
TV Dramas                         67
Thrillers                         65
Movies                            57
TV Action & Adventure             40
Stand-Up Comedy & Talk Shows      34
Romantic TV Shows                 32
Classic & Cult TV                 22
Anime Features  

In [16]:
df.iloc[8200:8210]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,duration_int,duration_type,primary_genre,content_age
8200,s8201,Movie,The Bachelors,Kurt Voelker,"J.K. Simmons, Josh Wiggins, Julie Delpy, Odeya...",United States,2018-02-21,2017,TV-14,100 min,Dramas,"After the death of his wife, a teacher and his...",2018.0,2.0,100.0,False,Dramas,9
8201,s8202,Movie,The Bad Education Movie,Elliot Hegarty,"Jack Whitehall, Joanna Scanlan, Iain Glen, Eth...",United Kingdom,2018-12-15,2015,TV-MA,87 min,Comedies,Britain's most ineffective but caring teacher ...,2018.0,12.0,87.0,False,Comedies,11
8202,s8203,Movie,The Bad Kids,"Keith Fulton, Louis Pepe",Unknown,United States,2017-04-01,2016,TV-MA,101 min,Documentaries,"In this documentary, teachers at a Mojave Dese...",2017.0,4.0,101.0,False,Documentaries,10
8203,s8204,Movie,The Bare-Footed Kid,Johnnie To,"Aaron Kwok, Lung Ti, Maggie Cheung, Chien-lien...",Hong Kong,2018-08-16,1993,TV-14,83 min,"Action & Adventure, International Movies","While working at a family friend's business, a...",2018.0,8.0,83.0,False,Action & Adventure,33
8204,s8205,Movie,The Basement,"Brian M. Conley, Nathan Ives","Mischa Barton, Jackson Davis, Cayleb Long, Tra...",United States,2019-04-09,2018,TV-MA,89 min,"Horror Movies, Thrillers",Known for tortuous role-play with victims in h...,2019.0,4.0,89.0,False,Horror Movies,8
8205,s8206,Movie,The Battle of Midway,John Ford,"Henry Fonda, Jane Darwell",United States,2017-03-31,1942,TV-14,18 min,"Classic Movies, Documentaries",Director John Ford captures combat footage of ...,2017.0,3.0,18.0,False,Classic Movies,84
8206,s8207,TV Show,The Beat,Unknown,"Henley Hii, Yise Loo, Aric Ho, Tiffany Leong, ...",Unknown,2017-09-18,2012,TV-MA,1 Season,"International TV Shows, TV Dramas",The artistic pursuits of two childhood friends...,2017.0,9.0,1.0,True,International TV Shows,14
8207,s8208,TV Show,The Beginning of Life: The Series,Estela Renner,Unknown,Brazil,2016-11-11,2016,TV-PG,1 Season,"Docuseries, International TV Shows, Science & ...",Using breakthroughs in technology and neurosci...,2016.0,11.0,1.0,True,Docuseries,10
8208,s8209,TV Show,The Bible's Buried Secrets,Unknown,Francesca Stavrakopoulou,United Kingdom,2019-02-01,2011,TV-14,1 Season,"British TV Shows, Docuseries, Science & Nature TV",Host Francesca Stavrakopoulou travels across t...,2019.0,2.0,1.0,True,British TV Shows,15
8209,s8210,TV Show,The Big Catch,Unknown,Ben Fogle,United Kingdom,2019-02-01,2015,TV-PG,1 Season,"British TV Shows, Reality TV","In a global competition, eight fishing enthusi...",2019.0,2.0,1.0,True,British TV Shows,11


In [17]:
df[df['show_id'].astype(str).str.contains('and probably', na=False)]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,duration_int,duration_type,primary_genre,content_age
